# Immortalite — Self-Play Training (Colab)

Trains the lightweight self-play chess engine on a free GPU. Checkpoints are written to Google Drive so the run survives disconnects.

**Steps:** set runtime to GPU (Runtime -> Change runtime type -> GPU), then run the cells top to bottom.

In [ ]:
# 1. Get the code. Safe to re-run (always works from /content, never nests).
import os
%cd /content
if not os.path.exists('/content/immortalite'):
    !git clone https://github.com/carlo-wong/immortalite.git
%cd /content/immortalite
!git pull --quiet
print('working dir:', os.getcwd())

In [ ]:
# 2. Install dependencies (Colab already has a CUDA torch build).
!pip install -q python-chess numpy

In [ ]:
# 3. Mount Google Drive for crash-proof checkpoints.
from google.colab import drive
drive.mount('/content/drive')
CKPT_DIR = '/content/drive/MyDrive/immortalite_checkpoints'
import os; os.makedirs(CKPT_DIR, exist_ok=True)
print('checkpoints ->', CKPT_DIR)

In [ ]:
# 4. Confirm GPU is available.
import torch
print('CUDA available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# 5. Train with the GPU preset. Resumes automatically from the Drive checkpoint.
import os
resume = os.path.join(CKPT_DIR, 'latest.pt')
resume_arg = f'--resume {resume}' if os.path.exists(resume) else ''
!python -m engine.train --iterations 50 --device cuda --gpu --checkpoint-dir "$CKPT_DIR" $resume_arg

In [ ]:
# 6. Plot training progress (run AFTER at least one training iteration finishes).
import csv, os
import matplotlib.pyplot as plt

path = os.path.join(CKPT_DIR, 'metrics.csv')
if not os.path.exists(path):
    print('No metrics.csv yet. Wait for iteration 0 of cell 5 to finish '
          '(a few minutes on the --gpu preset), then re-run this cell.')
else:
    its, pol, val = [], [], []
    with open(path) as f:
        for row in csv.DictReader(f):
            its.append(int(row['iter'])); pol.append(float(row['policy_loss'])); val.append(float(row['value_loss']))
    if not its:
        print('metrics.csv exists but is empty — wait for an iteration to finish.')
    else:
        fig, ax1 = plt.subplots(figsize=(8, 4))
        ax1.plot(its, pol, 'b-', label='policy loss'); ax1.set_xlabel('iteration'); ax1.set_ylabel('policy loss', color='b')
        ax2 = ax1.twinx(); ax2.plot(its, val, 'r-', label='value loss'); ax2.set_ylabel('value loss', color='r')
        plt.title('Immortalite training'); plt.show()

## After training
Download `latest.pt` from your Drive folder and run the analysis server locally:

```bash
set IMMORTALITE_CHECKPOINT=path\to\latest.pt   # Windows
python -m uvicorn server.app:app --port 8000
```
Then open http://localhost:8000/app/ .